## Text Analysis

### Setup

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import re #regular expressions
import nltk #natural language processing

Text and Data Cleaning

In [60]:
df=pd.DataFrame([['No'],['Yes'],['NO'],['YES'],['yes']], columns=['Yes or no?'])
df

,Yes or no?
0,No
1,Yes
2,NO
3,YES
4,yes


In [61]:
pd.get_dummies(df)


,Yes or no?_NO,Yes or no?_No,Yes or no?_YES,Yes or no?_Yes,Yes or no?_yes
0,False,True,False,False,False
1,False,False,False,True,False
2,True,False,False,False,False
3,False,False,True,False,False
4,False,False,False,False,True


Use string-cleaning tools to put into more usable format

In [62]:
df2=(df['Yes or no?'].copy()
            .str.title()
            )           
pd.get_dummies(df2)


,No,Yes
0,True,False
1,False,True
2,True,False
3,False,True
4,False,True


### Putting Regex to use

In [63]:
string="""Thank you for emailing chris@dogs.edu. I am out of the office. 
If you need to reach someone about our Big Barks Obedience School, 
please contact gabriela2@dogs.edu. For billing, contact ibrahim@accountant.com. 
For all other inquiries, please contact kiaan@dogs.edu."""
print(string)


Thank you for emailing chris@dogs.edu. I am out of the office. 
If you need to reach someone about our Big Barks Obedience School, 
please contact gabriela2@dogs.edu. For billing, contact ibrahim@accountant.com. 
For all other inquiries, please contact kiaan@dogs.edu.


In [64]:
emails=re.findall('\w+@\w+\.\w{3}', string)
print(emails)


['chris@dogs.edu', 'gabriela2@dogs.edu', 'ibrahim@accountant.com', 'kiaan@dogs.edu']


In [65]:
df=pd.DataFrame([['Ms. Jones', '123 South Drive', '91948'],
                ['Dr. Smith', '92 Main St.', '91762-1467'],
                ['Mrs. Pérez', '13579 Figueroa Blvd.', '98765-9287'],
                ['Mr. Lâm', '27987 Exposition Boulevard', 'I think it is 92341 but I moved recently']],
               columns=['Name','Address', 'Zip Code'])
 

In [66]:
df

,Name,Address,Zip Code
0,Ms. Jones,123 South Drive,91948
1,Dr. Smith,92 Main St.,91762-1467
2,Mrs. Pérez,13579 Figueroa Blvd.,98765-9287
3,Mr. Lâm,27987 Exposition Boulevard,I think it is 92341 but I moved recently


In [67]:
df['Last Name']=df['Name'].str.replace('^\w*\.\s','', regex=True)
df


,Name,Address,Zip Code,Last Name
0,Ms. Jones,123 South Drive,91948,Jones
1,Dr. Smith,92 Main St.,91762-1467,Smith
2,Mrs. Pérez,13579 Figueroa Blvd.,98765-9287,Pérez
3,Mr. Lâm,27987 Exposition Boulevard,I think it is 92341 but I moved recently,Lâm


In [68]:
df['zip5']=df['Zip Code'].str.extract('(\d{5})')
df

,Name,Address,Zip Code,Last Name,zip5
0,Ms. Jones,123 South Drive,91948,Jones,91948
1,Dr. Smith,92 Main St.,91762-1467,Smith,91762
2,Mrs. Pérez,13579 Figueroa Blvd.,98765-9287,Pérez,98765
3,Mr. Lâm,27987 Exposition Boulevard,I think it is 92341 but I moved recently,Lâm,92341


In [69]:
df=pd.DataFrame([[20908, 2341, 9179, 1, 0, 0, 0, 0],
                [28028, 1127, 7651, 0, 0, 1, 0, 0],
                [9928, 1878, 8769, 0, 1, 0, 0, 0],
                [10239, 2109, 8234, 0, 0, 0, 1, 0],
                [13027, 1356, 6750, 0, 0, 0, 0, 1]],   
               columns=['apples', 'bananas', 'pears','year2015',
                        'year2016','year2017','year2018','year2019'])
df

,apples,bananas,pears,year2015,year2016,year2017,year2018,year2019
0,20908,2341,9179,1,0,0,0,0
1,28028,1127,7651,0,0,1,0,0
2,9928,1878,8769,0,1,0,0,0
3,10239,2109,8234,0,0,0,1,0
4,13027,1356,6750,0,0,0,0,1


In [70]:
df.filter(regex='[a-zA-Z]+$')

,apples,bananas,pears
0,20908,2341,9179
1,28028,1127,7651
2,9928,1878,8769
3,10239,2109,8234
4,13027,1356,6750


### Use parantheses to search for groups of things

In [71]:
string = "Chris's email address chris@site.com."
matches=re.search(r'([A-Z]\w+).+(\b\w+@\w+\.\w+)', string)
print(matches.groups())
print(matches.group(1))
print(matches.group(2))


('Chris', 'chris@site.com')
Chris
chris@site.com


Bag-of-Words Application

In [72]:
df=pd.read_csv('data/bike_lane_comments.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'data/bike_lane_comments.csv'

See example comment

In [ ]:
print(df['Comment'][50])

In [ ]:
# A Python list where to store the tokenized words from all comments.
wordlist=[]
# Loop over the comments, one row at a time
for row in range(0,len(df)):
    
    # use regular expressions to keep list of words, in lowercase, no punctuation
    newlist=re.sub("[^a‑zA‑Z0‑9\s]", "", df['Comment'][row]).lower().split()
    
    # The "set" function will discard list ordering and delete duplicate words.
    new_words = list(set(newlist)‑set(wordlist))
    
    # Add any new words to the wordlist 
    wordlist=wordlist+new_words

len(wordlist)

Replicate transformation, but without splitting

In [ ]:
comments_transformed=(df['Comment'].copy()
                      .str.lower()
                      .str.replace("[^a-zA-Z0-\s]", "", regex=True)
                     )
comments_transformed[0:5]


Create new dataframe with one column per word and the same number of rows as our comments data

In [ ]:
df1=pd.DataFrame(np.zeros((len(df),len(wordlist))), columns=wordlist)


Count number of times each word appears in each comment

In [ ]:
for w in wordlist:
    df1[w]=comments_transformed.str.count(w)


Add word counts to our main dataframe

In [ ]:
df=pd.concat((df,df1),axis=1)
df.head()

### 1 Fit a logit with aggressive regularization and see which words are associated with which opinions

In [ ]:
from sklearn.linear_model import LogisticRegression
# Define and fit a machine learning lasso‑logit
# that will choose which words predict bike lane opinion well
bike_lane_logit=LogisticRegression(penalty='l1',
                                  random_state=0,
                                  solver='liblinear',
                                  C=0.1)
# Fit the logit
bike_lane_model=bike_lane_logit.fit(df[wordlist],df['Opposed'])


In [ ]:
# Extract the ﴾biased!﴿ logit coefficients
# The "flatnonzero" numpy function will give us a numpy
# array with the index number for each variable with
# a nonzero coefficient
support = np.flatnonzero(bike_lane_model.coef_)
print(support)

See which words predict bike lane opinion in lasso logit

In [ ]:
for i in support:
    print(wordlist[i]+" ‑ Coeff. "+str(round(bike_lane_model.coef_[0,i], 2)))


## Stop Words

Remove Words unnecessary for analysis

In [ ]:
from nltk.corpus import stopwords

## Rare Words

In [ ]:
rare_words = df[wordlist].sum(axis=0)
rare_words=rare_words[rare_words<=2]
rare_words

## Lemmatizing

In [ ]:
from nltk.stem.wordnet import WordNetLemmatizer as wnl

In [ ]:
wnl().lemmatize('mouse')

'mouse'

In [ ]:
wnl().lemmatize('mice')

'mouse'

In [ ]:
wnl().lemmatize('mouses')

'mouse'

In [ ]:
wnl().lemmatize("speak", pos='v') #part of speech optional argument

'speak'

In [ ]:
print(wnl().lemmatize("is", pos='v'))
print(wnl().lemmatize("are", pos='v'))
print(wnl().lemmatize("were", pos='v'))
print(wnl().lemmatize("was", pos='v'))

be
be
be
be
